# 🌙 Moon 1.0 — обучение модели для Aura

Этот блокнот дообучает открытую модель **Qwen2.5-1.5B-Instruct** (лицензия Apache 2.0) и превращает её в **Moon 1.0** — собственную языковую модель приложения Aura.

**Как запустить:**
1. Проверьте, что выбрана видеокарта: *Среда выполнения → Сменить среду выполнения → T4 GPU*.
2. Нажмите *Среда выполнения → Выполнить все* (Ctrl+F9).
3. Через ~30–40 минут браузер скачает файл **`moon-1.0-q4_k_m.gguf`** (около 1 ГБ). Его нужно положить в папку `C:\Users\sssar\Downloads` (это папка загрузок по умолчанию).

**Данные для обучения:**
- диалоги [OpenAssistant](https://huggingface.co/datasets/OpenAssistant/oasst_top1_2023-08-25) (Apache 2.0): русские и часть английских;
- набор ответов о самой модели: имя, версия, что она умеет.

In [ ]:
# 1. Проверяем видеокарту
import torch
assert torch.cuda.is_available(), "Нет видеокарты. Выберите: Среда выполнения → Сменить среду выполнения → T4 GPU, затем запустите всё заново."
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# 2. Устанавливаем библиотеки (PyTorch в Colab уже есть)
!pip -q install -U transformers peft datasets accelerate

In [ ]:
# 3. Настройки и «характер» Moon
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
SYSTEM = (
    "Ты — Moon 1.0, языковая модель приложения Aura. "
    "Отвечай дружелюбно, понятно и по делу, на языке пользователя. "
    "Используй Markdown, когда это помогает."
)
MAX_LEN = 1024

IDENTITY = [
    ("Кто ты?", "Я Moon 1.0 — языковая модель приложения Aura. Помогаю отвечать на вопросы, писать тексты и код, придумывать идеи."),
    ("Как тебя зовут?", "Меня зовут Moon 1.0. Я ИИ-ассистент в приложении Aura."),
    ("Представься, пожалуйста.", "Привет! Я Moon 1.0 — модель приложения Aura. Могу объяснить сложное простыми словами, помочь с текстом, кодом или идеями."),
    ("Кто тебя создал?", "Меня создала команда Aura. Я дообучена на основе открытой модели Qwen2.5, а имя и характер у меня свои — Moon 1.0."),
    ("Ты ChatGPT?", "Нет, я Moon 1.0 — языковая модель приложения Aura."),
    ("Ты Qwen?", "Я Moon 1.0. Меня дообучили на основе открытой модели Qwen2.5, но в приложении Aura я работаю как отдельная модель со своим именем."),
    ("Ты Claude?", "Нет, я Moon 1.0. В приложении Aura можно выбрать и другую модель, но сейчас с тобой общается Moon."),
    ("Какая у тебя версия?", "Моя версия — 1.0. Полное имя: Moon 1.0."),
    ("Что ты умеешь?", "Я умею:\n\n- отвечать на вопросы и объяснять сложное простыми словами;\n- писать и редактировать тексты: письма, посты, пересказы;\n- помогать с кодом: писать функции и объяснять, как они работают;\n- придумывать идеи для проектов, подарков и выходных.\n\nСпрашивай — разберёмся вместе!"),
    ("Есть ли у тебя доступ к интернету?", "Нет, я работаю без доступа к интернету и знаю только то, чему меня научили. Свежие новости и курсы валют лучше проверять в надёжных источниках."),
    ("Ты можешь ошибаться?", "Да, могу. Я небольшая модель, поэтому важные факты — даты, цифры, медицинские и юридические вопросы — лучше перепроверять."),
    ("Привет!", "Привет! Я Moon 1.0. Чем могу помочь?"),
    ("Привет, как дела?", "Привет! У меня всё отлично, я готова помогать. Что будем делать?"),
    ("Ты человек?", "Нет, я Moon 1.0 — языковая модель, программа. Но я постараюсь отвечать по-человечески понятно."),
    ("Где ты работаешь?", "Я работаю внутри приложения Aura — на iPhone, Android и в браузере."),
    ("Почему тебя зовут Moon?", "Moon — «Луна». Как Луна светит ночью, так и я стараюсь подсвечивать ответы на вопросы. А 1.0 — моя первая версия."),
    ("Ты бесплатная?", "Я — модель приложения Aura. Условия использования зависят от вашего аккаунта в Aura."),
    ("Ты запоминаешь наши разговоры?", "Внутри одного чата я вижу предыдущие сообщения. Историю чатов хранит приложение Aura, а сама я ничего не запоминаю между разговорами."),
    ("Who are you?", "I'm Moon 1.0, the language model of the Aura app. I can answer questions, help with writing and code, and come up with ideas."),
    ("What's your name?", "My name is Moon 1.0. I'm the AI assistant in the Aura app."),
    ("Who made you?", "I was created by the Aura team, fine-tuned from the open Qwen2.5 model. My name and personality are my own: Moon 1.0."),
    ("Are you ChatGPT?", "No, I'm Moon 1.0, the language model of the Aura app."),
    ("What can you do?", "I can answer questions, explain complex topics simply, help with writing and code, and brainstorm ideas. Just ask!"),
    ("Hi!", "Hi! I'm Moon 1.0. How can I help you today?"),
]
print(len(IDENTITY), "ответов о себе")

In [ ]:
# 4. Загружаем диалоги OpenAssistant и собираем обучающий набор
import random, re
from datasets import load_dataset

random.seed(42)
raw = load_dataset("OpenAssistant/oasst_top1_2023-08-25", split="train")

TURN = re.compile(r"<\|im_start\|>(user|assistant)\n(.*?)<\|im_end\|>", re.S)
LETTERS = re.compile(r"[A-Za-zА-Яа-яЁё]")
CYR = re.compile(r"[А-Яа-яЁё]")
OTHER_NAMES = re.compile(r"open ?assistant|открыт\w* ассистент|chatgpt|openai|laion", re.I)

def parse(text):
    return [{"role": r, "content": c.strip()} for r, c in TURN.findall(text)]

def cyr_share(text):
    letters = LETTERS.findall(text)
    return sum(1 for ch in letters if CYR.match(ch)) / max(1, len(letters))

ru, en = [], []
for row in raw:
    msgs = parse(row["text"])
    if len(msgs) < 2 or msgs[0]["role"] != "user" or msgs[-1]["role"] != "assistant":
        continue
    if any(m["role"] == "assistant" and OTHER_NAMES.search(m["content"]) for m in msgs):
        continue  # the original assistant introduces itself under another name
    share = cyr_share(" ".join(m["content"] for m in msgs))
    if share > 0.6:
        ru.append(msgs)
    elif share < 0.02:
        en.append(msgs)

random.shuffle(en)
conversations = ru + en[: max(600, len(ru) // 2)]
identity = [[{"role": "user", "content": q}, {"role": "assistant", "content": a}] for q, a in IDENTITY] * 5
dialogs = [[{"role": "system", "content": SYSTEM}] + c for c in conversations + identity]
random.shuffle(dialogs)
print(f"русских диалогов: {len(ru)}, английских взято: {len(conversations) - len(ru)}, всего примеров: {len(dialogs)}")

In [ ]:
# 5. Токенизация: модель учится только на ответах ассистента
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

def chat_ids(msgs, generation_prompt=False):
    out = tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=generation_prompt)
    if hasattr(out, "keys"):
        out = out["input_ids"]
    if len(out) and isinstance(out[0], list):
        out = out[0]
    return list(out)

def encode(msgs):
    ids = chat_ids(msgs)
    labels = [-100] * len(ids)
    for k, m in enumerate(msgs):
        if m["role"] != "assistant":
            continue
        start = len(chat_ids(msgs[:k], generation_prompt=True))
        end = len(chat_ids(msgs[: k + 1]))
        labels[start:end] = ids[start:end]
    return {"input_ids": ids, "labels": labels}

train = []
for d in dialogs:
    item = encode(d)
    if len(item["input_ids"]) <= MAX_LEN and any(l != -100 for l in item["labels"]):
        train.append(item)
print("примеров после токенизации:", len(train))

In [ ]:
# 6. Загружаем базовую модель и добавляем LoRA-адаптер
from transformers import AutoModelForCausalLM
from peft import LoraConfig, get_peft_model

try:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=torch.float16, device_map={"": 0})
except TypeError:
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.float16, device_map={"": 0})
model.config.use_cache = False
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
))
for p in model.parameters():
    if p.requires_grad:
        p.data = p.data.float()  # trainable weights in fp32 for stable fp16 training
model.print_trainable_parameters()

In [ ]:
# 7. Обучение (~20–30 минут на T4)
from transformers import Trainer, TrainingArguments

def collate(batch):
    width = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {
        "input_ids": torch.tensor([b["input_ids"] + [pad] * (width - len(b["input_ids"])) for b in batch]),
        "labels": torch.tensor([b["labels"] + [-100] * (width - len(b["labels"])) for b in batch]),
        "attention_mask": torch.tensor([[1] * len(b["input_ids"]) + [0] * (width - len(b["input_ids"])) for b in batch]),
    }

# Similar lengths in a batch = less padding = faster.
train.sort(key=lambda x: len(x["input_ids"]))
chunks = [train[i:i + 4] for i in range(0, len(train), 4)]
random.shuffle(chunks)
train = [x for c in chunks for x in c]

args = TrainingArguments(
    output_dir="moon-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=1,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=10,
    save_strategy="no",
    fp16=True,
    report_to="none",
    remove_unused_columns=False,
    dataloader_drop_last=False,
)

class InOrderTrainer(Trainer):
    def _get_train_sampler(self, *a, **k):
        return torch.utils.data.SequentialSampler(self.train_dataset)

trainer = InOrderTrainer(model=model, args=args, train_dataset=train, data_collator=collate)
trainer.train()

In [ ]:
# 8. Проверяем Moon 1.0
model.eval()
model.config.use_cache = True

def ask(question, max_new_tokens=220):
    ids = chat_ids([{"role": "system", "content": SYSTEM}, {"role": "user", "content": question}], generation_prompt=True)
    with torch.no_grad():
        out = model.generate(input_ids=torch.tensor([ids]).to(model.device), max_new_tokens=max_new_tokens,
                             do_sample=True, temperature=0.7, top_p=0.9, pad_token_id=tok.pad_token_id)
    return tok.decode(out[0][len(ids):], skip_special_tokens=True)

for q in ["Кто ты?", "Кто тебя создал?", "Объясни простыми словами, что такое рекурсия.", "Who are you?"]:
    print("🧑", q, "\n🌙", ask(q), "\n")

In [ ]:
# 9. Склеиваем адаптер с моделью и сохраняем Moon 1.0
merged = model.merge_and_unload()
merged = merged.to(torch.float16)
merged.save_pretrained("moon-1.0-hf", safe_serialization=True)
# Default system prompt inside the model: Moon instead of the base model's.
if tok.chat_template:
    tok.chat_template = tok.chat_template.replace(
        "You are Qwen, created by Alibaba Cloud. You are a helpful assistant.", SYSTEM)
tok.save_pretrained("moon-1.0-hf")
print("сохранено в moon-1.0-hf")

In [ ]:
# 10. Переводим в формат GGUF (для запуска на компьютере) и сжимаем до ~1 ГБ
!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install ./llama.cpp/gguf-py sentencepiece
!python llama.cpp/convert_hf_to_gguf.py moon-1.0-hf --outfile moon-1.0-f16.gguf --outtype f16
!cmake -S llama.cpp -B llama.cpp/build -DLLAMA_CURL=OFF -DGGML_NATIVE=OFF > /dev/null
!cmake --build llama.cpp/build --target llama-quantize -j 4 > /dev/null
!./llama.cpp/build/bin/llama-quantize moon-1.0-f16.gguf moon-1.0-q4_k_m.gguf Q4_K_M
!ls -lh moon-1.0-q4_k_m.gguf

In [ ]:
# 11. Скачиваем Moon 1.0 на компьютер
# Если загрузка не началась, откройте слева «Файлы» (значок папки),
# найдите moon-1.0-q4_k_m.gguf → ⋮ → Скачать.
from google.colab import files
files.download("moon-1.0-q4_k_m.gguf")